# Direct Gemini AI COICOP Product Classification

This notebook demonstrates direct zero-shot product classification using Google's active **`gemini-3.1-flash-lite`** model.

- **Input**: Real scraped product descriptions (food, snacks, bus tickets, kitchen utensils, personal hygiene, and Khmer script).
- **Process**: Sends a batch directly to the Gemini API with the UN COICOP taxonomy prompt.
- **Output**: Structured 4-digit COICOP Division and Class codes with category name, confidence, and reasoning.

## 1. Setup API Key & Gemini Client

In [ ]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types as genai_types

# Load API Key from .env
load_dotenv(dotenv_path="../.env")

raw_key = os.getenv("GEMINI_API_KEYS", os.getenv("GEMINI_API_KEY", ""))
api_key = [k.strip() for k in raw_key.split(",") if k.strip()][0]

# Use model configured in .env (gemini-3.1-flash-lite)
MODEL_NAME = os.getenv("GEMINI_MODEL", "gemini-3.1-flash-lite")
client = genai.Client(api_key=api_key)
print(f"Gemini API Client initialized! Active Model: {MODEL_NAME}")

## 2. Sample Products to Classify Directly
A diverse batch of real Cambodian retail products across different sectors (Grocery, Snacks, Transport, Household, Personal Care, and Khmer script).

In [ ]:
products_to_classify = [
    {"id": 1, "product_name": "SOUR WINEGUMS FOR CONNOISSEURS - CAVENDISH & HARVEY"},
    {"id": 2, "product_name": "SALTED CRACKERS - BOUTON D'OR"},
    {"id": 3, "product_name": "BUS TICKET: PHNOM PENH - SIEM REAP (VIP VAN)"},
    {"id": 4, "product_name": "HONEY SALTED ALMONDS - RITTER SPORT"},
    {"id": 5, "product_name": "NYLON KITCHEN SPATULA COOKING UTENSIL"},
    {"id": 6, "product_name": "COCA-COLA ORIGINAL TASTE 330ML CAN"},
    {"id": 7, "product_name": "DOVE SENSITIVE SKIN BAR SOAP 100G"},
    {"id": 8, "product_name": "GASOLINE SUPER 95 - CALTEX 1 LITER"},
    {"id": 9, "product_name": "សាប៊ូបោកខោអាវ ATTACK LAUNDRY DETERGENT 1KG"},
    {"id": 10, "product_name": "SAMSUNG GALAXY A27 5G 128GB BLACK"}
]

pd.DataFrame(products_to_classify)

## 3. Direct Gemini AI Classification Call
We send the batch of products directly to Gemini with instructions to output structured JSON.

In [ ]:
SYSTEM_PROMPT = """You are an expert UN COICOP (Classification of Individual Consumption According to Purpose) classification engine for the Consumer Price Index (CPI) in Cambodia.

Classify each product strictly into its official 4-digit COICOP Class code (format DD.G.C):
- 01: Food and non-alcoholic beverages (e.g. 01.1.1 Bread/Cereals, 01.1.8 Confectionery/Snacks, 01.2.2 Soft drinks)
- 02: Alcoholic beverages and tobacco (e.g. 02.1.3 Beer, 02.2.0 Tobacco)
- 03: Clothing and footwear
- 04: Housing, water, electricity, gas and other fuels
- 05: Furnishings and household maintenance (e.g. 05.4.1 Tableware/Kitchen utensils, 05.6.1 Cleaning goods)
- 06: Health (e.g. 06.1.1 Medicines)
- 07: Transport (e.g. 07.2.2 Fuels, 07.3.2 Passenger road transit / Bus tickets)
- 08: Communication (e.g. 08.2.0 Mobile phones, 08.3.0 Internet)
- 09: Recreation and culture
- 10: Education
- 11: Restaurants and hotels
- 12: Miscellaneous goods and personal care (e.g. 12.1.3 Soap/Hygiene)

Return strict JSON list only:
[
  {
    "id": <int>,
    "coicop_division": "<2-digit division e.g. 01>",
    "coicop_code": "<4-digit class code e.g. 01.1.8>",
    "category_name": "<category title>",
    "confidence": <float 0.0-1.0>,
    "reason": "<short reason under 8 words>"
  }
]
"""

prompt_input = f"Classify these products:\n{json.dumps(products_to_classify, ensure_ascii=False, indent=2)}"

# Call active Gemini Flash model directly
response = client.models.generate_content(
    model=MODEL_NAME,
    contents=prompt_input,
    config=genai_types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT,
        response_mime_type="application/json",
        temperature=0.0
    )
)

# Parse response and display
results = json.loads(response.text)
df_classified = pd.DataFrame(results)

# Merge with input products for clear display
df_final = pd.DataFrame(products_to_classify).merge(df_classified, on="id")
df_final[["id", "product_name", "coicop_division", "coicop_code", "category_name", "confidence", "reason"]]